# Week 5 · RoPE, YaRN & Long-Context Extension

**Repository:** `bitwise-llm-forge` · **Theory doc:** [`docs/theory/week5_rope.md`](../docs/theory/week5_rope.md)

---

### Learning objectives

1. Derive **Rotary Position Embedding** as a complex-plane rotation $z_i \mapsto z_i e^{im\theta_i}$.
2. Prove the **relative-position property**: $\langle f(q,m), f(k,n)\rangle$ depends only on $m-n$.
3. Build the standard and **NTK-aware (YaRN)** variants and inspect their frequency tables.
4. Empirically demonstrate that naive RoPE *fails* past its training context, while YaRN extends gracefully.
5. Plot perplexity-vs-position curves on a controlled synthetic sequence task.


In [ ]:
# Make ``src/`` importable when this notebook is launched from anywhere.
from pathlib import Path
import sys

_here = Path.cwd()
for candidate in (_here, *_here.parents):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from src.utils.seeding import set_seed
set_seed(42)

print("environment ready · torch", __import__("torch").__version__)


## 1 · Mathematical derivation

### 1.1 RoPE as a complex rotation

Split the $d$-dimensional embedding into $d/2$ complex numbers
$z_i = x_{2i} + i\,x_{2i+1}$. With per-pair frequency

$$\theta_i = b^{-2i/d}, \qquad b = 10{,}000,$$

the RoPE map at position $m$ is multiplication by the unit complex number

$$f(z_i, m) = z_i \cdot e^{i m \theta_i}.$$

In real form this is the standard 2D rotation matrix.

### 1.2 Relative-position property

For two positions $m, n$:

$$
\langle f(q,m), f(k,n)\rangle \;=\; \operatorname{Re}\sum_i q_i \bar k_i\, e^{i(m-n)\theta_i}
$$

— a function of **only** the offset $m-n$.

### 1.3 Why bare RoPE breaks past training context

Each dimension pair rotates with period $2\pi/\theta_i$.

- **High-frequency dims** ($i$ small): period ≈ a few tokens → many wraps within the training window.
- **Low-frequency dims** ($i \approx d/2$): period ≈ $2\pi b$ ≈ 62k tokens for $b{=}10^4$ → the model only sees the first fraction of the cycle.

Extrapolating to $N\gg L_\text{train}$ pushes the low-frequency dims into **unseen phase territory** → perplexity explodes.

### 1.4 NTK-aware interpolation (YaRN)

YaRN partitions dimensions by wavelength:

- $\lambda_i < L_\text{train}/\alpha$: leave $\theta_i$ alone (the dim already wraps).
- $\lambda_i > L_\text{train}\cdot\beta$: linearly interpolate to $\theta_i / s$ where $s = L_\text{target}/L_\text{train}$.
- In between: smooth ramp.


## 2 · Reference implementations

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.positional import RotaryPositionalEmbedding, YaRNRotaryPositionalEmbedding

torch.manual_seed(0)

dim, max_seq = 64, 4096
rope = RotaryPositionalEmbedding(dim=dim, max_seq_len=max_seq, base=10_000.0)
yarn = YaRNRotaryPositionalEmbedding(
    dim=dim,
    max_seq_len=max_seq * 8,                # extend 8×
    original_max_seq_len=max_seq,
    scale_factor=8.0,
    base=10_000.0,
)

# A tensor of shape (B, H, N, D) is what a transformer would call rope() on.
x = torch.randn(1, 1, 128, dim)
y = rope(x)
print("input :", tuple(x.shape), "  output:", tuple(y.shape))
print("RoPE preserves norms?  max ratio diff:",
      float((y.norm(dim=-1) / x.norm(dim=-1) - 1).abs().max().item()))


## 3 · Sanity check: the relative-position property

In [ ]:
# For random q, k and any offset δ, <rope(q, m), rope(k, m+δ)>
# should be (approximately) the same as <rope(q, m'), rope(k, m'+δ)>.

torch.manual_seed(0)
q = torch.randn(1, 1, 1, dim)
k = torch.randn(1, 1, 1, dim)

def rope_at(x, pos: int) -> torch.Tensor:
    return rope(x, position_ids=torch.tensor([pos]))


offsets   = [0, 1, 4, 16, 64]
abs_pairs = [(10, 20), (100, 200), (300, 500)]

print(f"{'offset':>8}  " + "  ".join(f"<q@{a:>4},k@{b:>4}>" for a, b in abs_pairs))
for δ in offsets:
    vals = []
    for a, _ in abs_pairs:
        qm = rope_at(q, a)
        kn = rope_at(k, a + δ)
        vals.append(float((qm * kn).sum().item()))
    print(f"  δ={δ:>4}  " + "  ".join(f"{v:>14.6f}" for v in vals))

print("\n→ values in each row are equal (up to fp32 noise), confirming the relative-position property.")


## 4 · Inspecting the YaRN frequency rescaling

In [ ]:
# Plot the inverse frequencies for plain RoPE vs YaRN.
inv_rope = rope.inv_freq.detach().numpy()
inv_yarn = yarn.inv_freq.detach().numpy()
xs = list(range(len(inv_rope)))

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(xs, inv_rope, marker="o", label="vanilla RoPE")
ax.plot(xs, inv_yarn, marker="s", label="YaRN (scale=8×)")
ax.set_xlabel("frequency index $i$ (0 = highest freq)")
ax.set_ylabel("inverse frequency $\\theta_i$")
ax.set_yscale("log"); ax.grid(True, which="both", alpha=0.3); ax.legend()
ax.set_title("YaRN preserves high frequencies, scales down low frequencies")
plt.tight_layout(); plt.show()

# Print first / last few entries
print("first 4 freqs   (kept by YaRN):")
for i in range(4):
    print(f"  i={i}  rope={inv_rope[i]:.5f}   yarn={inv_yarn[i]:.5f}")
print("last 4 freqs   (rescaled by YaRN):")
for i in range(len(inv_rope) - 4, len(inv_rope)):
    print(f"  i={i}  rope={inv_rope[i]:.5e}   yarn={inv_yarn[i]:.5e}")


## 5 · A controlled long-context perplexity experiment

We need a model where we can change the positional encoding *at evaluation time* without retraining, and a sequence task with non-trivial position dependence.

**Setup.** Train a tiny 2-layer transformer (with RoPE) on a synthetic next-token task at context length 256. Then evaluate the trained model at extended contexts (512, 1024, 2048) with three positional schemes:

1. **Vanilla RoPE** (the one it was trained with).
2. **Position-Interpolation (PI)** — linearly compress positions back into the train range.
3. **YaRN** — NTK-aware interpolation.

The expected result: vanilla RoPE's perplexity explodes; PI and YaRN stay finite, with YaRN typically winning.


In [ ]:
# A small, fast transformer for the experiment.
class MiniRoPETransformer(nn.Module):
    def __init__(self, vocab: int, d_model: int = 64, n_heads: int = 4,
                 n_layers: int = 2, max_seq_len: int = 256):
        super().__init__()
        self.d_model = d_model; self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.embed = nn.Embedding(vocab, d_model)
        self.rope: RotaryPositionalEmbedding = RotaryPositionalEmbedding(
            dim=self.head_dim, max_seq_len=max_seq_len)
        self.blocks = nn.ModuleList([
            nn.ModuleDict({
                "ln1":  nn.LayerNorm(d_model),
                "qkv":  nn.Linear(d_model, 3 * d_model, bias=False),
                "proj": nn.Linear(d_model, d_model, bias=False),
                "ln2":  nn.LayerNorm(d_model),
                "ffn":  nn.Sequential(
                    nn.Linear(d_model, 4 * d_model), nn.GELU(),
                    nn.Linear(4 * d_model, d_model)),
            }) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, ids):
        b, n = ids.shape
        x = self.embed(ids)
        for blk in self.blocks:
            h = blk["ln1"](x)
            qkv = blk["qkv"](h).reshape(b, n, 3, self.n_heads, self.head_dim)
            q, k, v = qkv.unbind(dim=2)
            q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)
            q = self.rope(q); k = self.rope(k)
            a = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
            a = a.transpose(1, 2).reshape(b, n, self.d_model)
            x = x + blk["proj"](a)
            x = x + blk["ffn"](blk["ln2"](x))
        return self.head(self.norm(x))


# Synthetic task: predict the next character of a simple periodic stream.
# This is hard enough to require positional information but small enough
# to train in a few hundred steps.
vocab = 32

def make_stream(n: int, vocab: int, periods=(7, 11, 13)) -> torch.Tensor:
    pos = torch.arange(n)
    base = sum((pos // p) % (vocab // 3) for p in periods)
    noise = torch.randint(0, vocab // 8, (n,))
    return ((base + noise) % vocab).long()


train_stream = make_stream(60_000, vocab)
print("training stream length:", len(train_stream))


In [ ]:
from src.utils.seeding import set_seed
set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

train_ctx = 256
model = MiniRoPETransformer(vocab=vocab, max_seq_len=train_ctx).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

stream = train_stream.to(device)

steps = 600 if device.type == "cuda" else 250
batch = 32
print(f"training for {steps} steps with batch={batch}, ctx={train_ctx}...")

losses = []
for step in range(steps):
    starts = torch.randint(0, len(stream) - train_ctx - 1, (batch,))
    idx = starts[:, None] + torch.arange(train_ctx + 1)[None, :]
    chunk = stream[idx]
    ids, targets = chunk[:, :-1], chunk[:, 1:]

    logits = model(ids)
    loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, vocab), targets.reshape(-1))
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

    if step % 50 == 0:
        losses.append((step, float(loss.item())))
        print(f"  step {step:4d}  loss {loss.item():.4f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot([x for x, _ in losses], [y for _, y in losses], marker="o")
ax.set_xlabel("step"); ax.set_ylabel("CE loss")
ax.set_title(f"Training loss (context {train_ctx})")
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# Build a longer eval stream and three positional-encoding variants.
eval_stream = make_stream(40_000, vocab).to(device)
eval_ctxs = [256, 512, 1024, 2048]


def replace_rope(model, new_rope):
    """Swap the model's RoPE module in-place."""
    model.rope = new_rope.to(device)
    return model


@torch.no_grad()
def perplexity(model, ctx: int, n_windows: int = 16) -> float:
    starts = torch.randint(0, len(eval_stream) - ctx - 1, (n_windows,))
    total_loss, total_tok = 0.0, 0
    for s in starts:
        chunk = eval_stream[s:s + ctx + 1].unsqueeze(0)
        logits = model(chunk[:, :-1])
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, vocab),
            chunk[:, 1:].reshape(-1), reduction="sum")
        total_loss += float(loss.item())
        total_tok  += chunk.shape[1] - 1
    return math.exp(total_loss / total_tok)


# Three positional-encoding variants
head_dim = model.head_dim

rope_vanilla = RotaryPositionalEmbedding(
    dim=head_dim, max_seq_len=max(eval_ctxs), base=10_000.0)

# Position interpolation: shorten max_seq_len so position_ids get implicitly
# rescaled (we approximate by using a smaller effective scale).
class PositionInterpolation(RotaryPositionalEmbedding):
    def __init__(self, dim, max_seq_len, train_seq_len, base=10_000.0):
        super().__init__(dim=dim, max_seq_len=max_seq_len, base=base)
        self.factor = train_seq_len / max_seq_len
    def forward(self, x, position_ids=None):
        seq_len = x.shape[-2]
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=x.device)
        # rescale positions back into the train range
        return super().forward(x, position_ids=(position_ids.float() * self.factor).long().clamp(max=self.max_seq_len-1))


rope_pi = PositionInterpolation(
    dim=head_dim, max_seq_len=max(eval_ctxs), train_seq_len=train_ctx)

rope_yarn = YaRNRotaryPositionalEmbedding(
    dim=head_dim, max_seq_len=max(eval_ctxs),
    original_max_seq_len=train_ctx,
    scale_factor=max(eval_ctxs) / train_ctx,
    base=10_000.0,
)

variants = {
    "Vanilla RoPE": rope_vanilla,
    "Position Interp.": rope_pi,
    "YaRN":          rope_yarn,
}

results = {name: [] for name in variants}
for name, r in variants.items():
    replace_rope(model, r)
    print(f"\n{name}:")
    for ctx in eval_ctxs:
        ppl = perplexity(model, ctx)
        results[name].append((ctx, ppl))
        print(f"  ctx={ctx:>5}  ppl={ppl:.3f}")


In [ ]:
# Plot perplexity vs context length
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for name, rows in results.items():
    xs = [c for c, _ in rows]; ys = [p for _, p in rows]
    ax.plot(xs, ys, "-o", label=name, linewidth=2)
ax.axvline(train_ctx, color="grey", linestyle=":", alpha=0.7, label=f"train context = {train_ctx}")
ax.set_xlabel("evaluation context length")
ax.set_ylabel("perplexity")
ax.set_yscale("log")
ax.set_xscale("log", base=2)
ax.set_xticks(eval_ctxs); ax.set_xticklabels([str(c) for c in eval_ctxs])
ax.set_title("Long-context extrapolation: RoPE vs PI vs YaRN")
ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Persist
out = {name: [{"ctx": c, "ppl": p} for c, p in rows] for name, rows in results.items()}
Path("../benchmarks/results").mkdir(parents=True, exist_ok=True)
Path("../benchmarks/results/rope_extrapolation.json").write_text(json.dumps(out, indent=2))
print("\nresults saved to ../benchmarks/results/rope_extrapolation.json")


## 6 · Discussion

- **Vanilla RoPE breaks at the training boundary** — the curve typically jumps sharply at $N > L_\text{train}$. The model has never seen the relevant low-frequency phases.
- **PI is the cheap fix; YaRN is the principled one.** PI globally compresses all frequencies, which damages high-frequency discrimination. YaRN preserves high frequencies untouched and only rescales low ones — better at extreme extensions.
- **No fine-tuning needed (at small scales).** The plot above is from a *zero-shot* substitution at eval time. At real-LLM scale a brief context-extension fine-tune (~1k steps) closes the remaining gap.
- **LongRoPE goes further.** Instead of a one-parameter scaling, LongRoPE searches over a per-dimension factor vector via evolutionary search, achieving 2M-token contexts.

### References
- Su, J. et al. "RoFormer: Enhanced Transformer with Rotary Position Embedding." arXiv:2104.09864, 2021.
- Chen, S. et al. "Extending Context Window of Large Language Models via Positional Interpolation." arXiv:2306.15595, 2023.
- Peng, B. et al. "YaRN: Efficient Context Window Extension of Large Language Models." arXiv:2309.00071, 2023.
- Ding, Y. et al. "LongRoPE: Extending LLM Context Window Beyond 2 Million Tokens." 2024.
